# steerbench — quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bamdadd/steerbench/blob/main/notebooks/steerbench_quickstart.ipynb)

> **The steering-vector report card.** From a fresh Colab to a rendered report card in well under 10 minutes — **no GPU, no model download.**

This notebook renders the *committed M0 sweep artifacts* — real `Qwen2.5` formality-vector runs — so the dose-response cliff and the layer plateau you see are genuine measurements, not a toy. The heavy GPU sweep already ran; here we only parse its CSVs and render the card on CPU.

**What you get**
1. **Dose-response** — effect vs coherence across injection strength (the hero figure, with the sweet spot and the coherence cliff marked).
2. **Effect size**, **side effects**, and **layer sensitivity** — the full four-part card, rendered inline.

## 1. Install

Clone the repo (for the committed sweep artifacts) and install steerbench with the `vectors`, `model`, and `report` extras. On Colab `torch` / `transformers` / `matplotlib` are preinstalled, so this is quick.

In [ ]:
!git clone --depth 1 https://github.com/bamdadd/steerbench.git
%cd steerbench
!pip install -q -e ".[vectors,model,report]"

## 2. (Optional) inspect the steering vector

The report card is built from sweep CSVs and needs **no vector**. If you have a repeng formality vector (`.gguf`), point `EXAMPLE_VECTOR` at it to see provenance and per-layer direction norms. Absent a staged vector, we fall back to the per-layer `dir_norm` already recorded in the sweep CSV, so the cell still shows real provenance.

In [ ]:
from pathlib import Path

EXAMPLE_VECTOR = Path('examples/formality.gguf')  # stage your own to load a real vector

if EXAMPLE_VECTOR.exists():
    from steerbench.vectors import load_vector

    vec = load_vector(EXAMPLE_VECTOR)
    print('concept :', vec.concept, '| source model:', vec.model_id)
    for layer, norm in sorted(vec.layer_norms().items()):
        print(f'  L{layer:<3d} \u2016dir\u2016 = {norm:.4f}')
else:
    import csv

    print('No example vector staged; showing per-layer direction norms from the sweep CSV.')
    norms: dict[int, float] = {}
    with open('artifacts/layer_sweep.csv') as fh:
        for row in csv.DictReader(fh):
            norms.setdefault(int(row['layer']), float(row['dir_norm']))
    for layer in sorted(norms):
        print(f'  L{layer:<3d} \u2016dir\u2016 = {norms[layer]:.4f}')

## 3. Build the report card

CPU-only: `steer-report` parses the dose-response and layer-sweep CSVs, analyses them (sweet spot, coherence cliff, coherent plateau, degenerate-trap layers), and writes markdown + a self-contained HTML card with the plots embedded.

In [ ]:
!steer-report \
    --dose-csv artifacts/dose_response.csv \
    --layer-csv artifacts/layer_sweep.csv \
    --out report_out --stem report

## 4. The hero — dose-response curve

Effect (blue) climbs with the injection coefficient while perplexity (red) holds flat, then falls off the **coherence cliff**. The sweet spot is the strongest push that stays fluent.

In [ ]:
from IPython.display import Image

Image('report_out/report_dose.png')

## 5. The full report card (rendered inline)

The self-contained HTML — all four panels, plots embedded as base64, no external assets — rendered directly in the notebook.

In [ ]:
from IPython.display import HTML

HTML(Path('report_out/report.html').read_text())

## Next steps

- **Your own vector:** drop a repeng `.gguf` at `examples/formality.gguf` and re-run cell 2 to inspect it.
- **Your own sweep:** the GPU/Modal harness lives in `experiments/modal_app.py` (`pip install steerbench[gpu]`), reached via `steer-report --run <function>`. It writes the same CSV schema this card consumes, so re-running cell 3 renders your results.
- **Another model:** run the sweep on Llama 3.x 8B / Gemma 2 9B and compare cards — steerability varies by architecture.